# Day 6 — Hyperparameter tuning with Optuna + SHAP


In [ ]:
import optuna
import numpy as np, pandas as pd, seaborn as sns
from sklearn.model_selection import StratifiedKFold, cross_val_score
from xgboost import XGBClassifier

df = sns.load_dataset('titanic').dropna(subset=['age', 'embarked'])
y = df['survived']
X = pd.get_dummies(df[['pclass','sex','age','sibsp','parch','fare','embarked']], columns=['sex','embarked'], drop_first=True)

cv = StratifiedKFold(5, shuffle=True, random_state=0)

def objective(trial):
    params = dict(
        n_estimators=trial.suggest_int('n_estimators', 100, 800),
        max_depth=trial.suggest_int('max_depth', 2, 8),
        learning_rate=trial.suggest_float('lr', 1e-3, 0.3, log=True),
        subsample=trial.suggest_float('subsample', 0.5, 1.0),
        colsample_bytree=trial.suggest_float('cb', 0.5, 1.0),
        reg_lambda=trial.suggest_float('lam', 1e-3, 10.0, log=True),
        eval_metric='logloss',
        random_state=0,
    )
    m = XGBClassifier(**params)
    return cross_val_score(m, X, y, scoring='f1', cv=cv, n_jobs=-1).mean()

study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=50, show_progress_bar=False)
print('best F1:', study.best_value)
print('best params:', study.best_params)


## SHAP explanations

In [ ]:
import shap
best = XGBClassifier(**{**study.best_params, 'eval_metric':'logloss', 'random_state':0,
                       'n_estimators': study.best_params['n_estimators'],
                       'learning_rate': study.best_params['lr']}).fit(X, y)
explainer = shap.TreeExplainer(best)
sv = explainer.shap_values(X)
shap.summary_plot(sv, X)
